# Quickstart
See the next tutorial, [Toy Example](toy_example/), for a full walkthrough.

See [Run Clumppling](run_clumppling/) for a guide to running *Clumppling* with ACE-OF-Clust.

See [Real Example](real_example/) for a demonstration on real data (PBMC3k scRNA-seq hard-clustering model comparison).

#### Setup

In [ ]:
import ace_of_clust as aoc
print(aoc.__version__ if hasattr(aoc, '__version__') else 'import ok')

#### Run *Clumppling*
Assume your clustering results are saved as `.Q` matrix files (one-hot encoded for hard clustering) under `cls_dir`, where each row contains K membership values that sum to 1 (K is the number of clusters) and all files have the same number of rows.

In [ ]:
align_dir = "path/to/clumppling/alignment_results"
cls_dir = "path/to/clumppling/clustering_results"
aoc.run_clumppling_via_main(
        input_dir=cls_dir,
        output_dir=align_dir,
        fmt="generalQ",                    
        vis=False,
        extension=".Q",                         
    )

#### Load *Clumppling* results

In [ ]:
results = aoc.load_clumppling_results(
    align_dir=align_dir,
    suffix="rep",
    cls_dir=cls_dir,
    load_P=True,
    strict_P=True,   # will raise FileNotFoundError if any P file is missing; set to False to skip loading missing P files
)

#### Compute pairwise mappings of clusters between modes

In [ ]:
pair_mappings = aoc.extract_all_mode_pair_mappings(
    mode_names=results.modes,
    all_modes_alignment=results.all_modes_alignment,
    alignment_acrossK=results.alignment_acrossK,
)

#### Compute feature metrics for all modes

In [ ]:
features = [...]  # list of features (gene) names used in Clumppling
df_by_mode = aoc.compute_feature_metrics_all_modes(results, feature_names=features)

#### Select top features (genes) by *weighted_Psum* quantile across modes 

In [ ]:
selected_by_mode, df_selected_all, overlap = aoc.select_top_features_by_weighted_Psum(
    df_by_mode,
    top_quantile=0.1,
)

#### Visualize cluster memberships
Assume that you have loaded some coordinates for your data points (cells/spots) in `X_coords` and have provided a list of colors to be used in `colors`.

In [ ]:
fig, axes = aoc.overlay_scatter_for_mode(
    results,
    coords=X_coords,
    cluster_colors=colors[:results.K_max],
    val_threshold=0.5,
    s=5, alpha=0.9,
    suptitle=f"Cluster Memberships",
    suptitle_kwargs= {'y':0.95, 'fontsize':10},
)